# GeneSense × MedOCR-Vision — 100-image LoRA experiment

This notebook starts from the published MedOCR-Vision checkpoint, adds a small LoRA adapter, and compares the same held-out images before/after training. It exports per-image CER/WER evidence. It does not diagnose or check medication safety.

**Privacy gate:** use only authorized, de-identified or synthetic images and labels. Colab runs on a Google-managed machine. Do not upload identifiable medical records unless authorization and your privacy/security requirements explicitly allow it. Notebook outputs avoid printing OCR text, but the exported CSV contains transcriptions and must remain private.

Prepare a ZIP with `manifest.csv` and `images/`; see the companion README and manifest template. For 100 images, aim for 70 train / 15 validation / 15 test. Keep groups/templates/patient pseudonyms within a single split.

In [ ]:
# Install the versions used by the upstream MedOCR training notebook.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → GPU, then rerun.')
%pip install -q 'transformers==4.56.2' 'trl==0.22.2' 'datasets==4.3.0' unsloth bitsandbytes accelerate peft einops jiwer pillow

In [ ]:
import torch, platform
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1024**3
print(f'Python {platform.python_version()} | PyTorch {torch.__version__}')
print(f'GPU {props.name} | VRAM {VRAM_GB:.1f} GB | BF16 {torch.cuda.is_bf16_supported()}')
print('Target is 24 GB VRAM; smaller GPUs may OOM even with 4-bit loading.')

In [ ]:
# Set True only after reviewing authorization and de-identification for every image and label.
DATA_DEIDENTIFIED = False
if not DATA_DEIDENTIFIED:
    raise RuntimeError('Review/de-identify the complete bundle, then explicitly set DATA_DEIDENTIFIED = True.')

from google.colab import files
from pathlib import Path
import zipfile, shutil
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
if len(zip_names) != 1: raise ValueError('Upload exactly one ZIP containing manifest.csv and images/.')
DATA_DIR = Path('/content/medocr_data')
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as zf:
    if any(Path(n).is_absolute() or '..' in Path(n).parts for n in zf.namelist()): raise ValueError('Unsafe path in ZIP.')
    zf.extractall(DATA_DIR)
manifest_paths = list(DATA_DIR.rglob('manifest.csv'))
if len(manifest_paths) != 1: raise ValueError('ZIP must contain exactly one manifest.csv.')
MANIFEST_PATH = manifest_paths[0]
print('Bundle unpacked; no image or transcription content was displayed.')

In [ ]:
import pandas as pd, hashlib
from PIL import Image
df = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False, encoding='utf-8')
required = {'filename','split','group_id','text'}
if not required.issubset(df.columns): raise ValueError(f'Missing columns: {sorted(required - set(df.columns))}')
if df.empty or df.filename.duplicated().any(): raise ValueError('Manifest is empty or has duplicate filenames.')
if (df.text.str.strip() == '').any() or (df.group_id.str.strip() == '').any(): raise ValueError('Every row needs non-empty text and anonymous group_id.')
if set(df.split) != {'train','validation','test'}: raise ValueError('Need non-empty train, validation, and test splits.')
if not set(df.split).issubset({'train','validation','test'}): raise ValueError('Invalid split name.')
if (df.groupby('group_id').split.nunique() > 1).any(): raise ValueError('Group leakage: a group_id occurs across splits.')
image_root = MANIFEST_PATH.parent / 'images'
if not image_root.is_dir(): raise ValueError('Expected images/ beside manifest.csv.')
seen = {}
for row in df.to_dict('records'):
    name = row['filename']; path = image_root / name
    if Path(name).name != name or path.suffix.lower() not in {'.jpg','.jpeg','.png'} or not path.is_file(): raise ValueError(f'Missing/unsupported image: {name}')
    with Image.open(path) as im: im.verify()
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest in seen and seen[digest] != row['split']: raise ValueError('Identical image bytes cross splits.')
    seen[digest] = row['split']; row['image_path'] = str(path)
df['image_path'] = [str(image_root / n) for n in df.filename]
counts = df.split.value_counts().to_dict()
print(f'Validated {len(df)} images; splits={counts}')
print('Recommended for 100: 70/15/15. Group by source/template before assigning splits.')

In [ ]:
# Model custom code is pinned to this reviewed Hub commit for repeatability.
MODEL_ID = 'naazimsnh02/medocr-vision'
MODEL_REVISION = '753b26267e816d156a4ec783b1f7071cf9ebb565'
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = VRAM_GB < 22
from unsloth import FastVisionModel
from transformers import AutoModelForCausalLM, AutoProcessor
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT, full_finetuning=False,
    auto_model=AutoModelForCausalLM, trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True)
print(f'Checkpoint loaded; 4-bit={LOAD_IN_4BIT}; GPU={torch.cuda.get_device_name(0)}')

In [ ]:
from PIL import Image
INSTRUCTION = 'Extract all visible text from this document. Preserve original wording, numbers, units, and order. Do not infer missing text.'
def to_conversation(row):
    image = Image.open(row['image_path']).convert('RGB')
    return {'images':[image], 'messages':[
        {'role':'user','content':[{'type':'text','text':INSTRUCTION},{'type':'image','image':image}]},
        {'role':'assistant','content':[{'type':'text','text':row['text']}]},
    ]}
records = {s:[to_conversation(r) for r in df[df.split == s].to_dict('records')] for s in ('train','validation','test')}
print({s:len(rows) for s,rows in records.items()})

In [ ]:
# Baseline evaluation on held-out test images before any training.
import unicodedata, re, time
from jiwer import cer, wer
def normalize_text(value):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(value)).casefold()).strip()
@torch.inference_mode()
def predict(image, active_model, active_processor, active_tokenizer, max_new_tokens=1024):
    messages = [{'role':'user','content':[{'type':'image'},{'type':'text','text':INSTRUCTION}]}]
    prompt = active_processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = active_processor(image, prompt, add_special_tokens=False, return_tensors='pt').to('cuda')
    prompt_len = inputs['input_ids'].shape[-1]
    output = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return active_tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()
test_rows = df[df.split == 'test'].to_dict('records')
baseline_predictions = []; t0 = time.time(); FastVisionModel.for_inference(model)
for row in test_rows:
    with Image.open(row['image_path']) as im: baseline_predictions.append(predict(im.convert('RGB'), model, processor, tokenizer))
baseline_seconds = time.time() - t0
refs = [normalize_text(r['text']) for r in test_rows]; baseline = [normalize_text(p) for p in baseline_predictions]
print(f'Baseline test n={len(refs)} CER={cer(refs, baseline):.4f} WER={wer(refs, baseline):.4f} time={baseline_seconds:.1f}s')

In [ ]:
# Small pilot settings; test remains held out.
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig, SFTTrainer
model = FastVisionModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.05, bias='none', random_state=3407, use_rslora=False,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj','out_proj','fc1','fc2','linear_1','linear_2'],
)
FastVisionModel.for_training(model)
collator = UnslothVisionDataCollator(model=model, processor=processor, ignore_index=-100, max_seq_length=MAX_SEQ_LENGTH,
    train_on_responses_only=True, instruction_part='User: ', response_part='\nAssistant:', pad_to_multiple_of=8)
bf16 = torch.cuda.is_bf16_supported()
train_args = SFTConfig(
    output_dir='/content/medocr_checkpoints', per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=2e-5, warmup_ratio=0.05,
    lr_scheduler_type='linear', optim='adamw_8bit', weight_decay=0.01, logging_steps=1,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model='eval_loss', greater_is_better=False, report_to='none', remove_unused_columns=False,
    dataset_text_field='', dataset_kwargs={'skip_prepare_dataset':True}, max_length=MAX_SEQ_LENGTH,
    fp16=not bf16, bf16=bf16, gradient_checkpointing=True, seed=3407)
trainer = SFTTrainer(model=model, tokenizer=processor.tokenizer, data_collator=collator,
    train_dataset=records['train'], eval_dataset=records['validation'], args=train_args)
print(f'Ready: train={len(records["train"])} validation={len(records["validation"])}; test is not passed to trainer.')

In [ ]:
train_result = trainer.train()
print(f'Train runtime: {train_result.metrics.get("train_runtime", 0):.1f}s | loss: {train_result.metrics.get("train_loss", float("nan")):.4f}')
print(f'Best validation loss: {trainer.state.best_metric}')
print(f'Peak GPU allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB')

In [ ]:
# Final comparison, exactly once on held-out test.
FastVisionModel.for_inference(model); t0 = time.time(); tuned_predictions = []
for row in test_rows:
    with Image.open(row['image_path']) as im: tuned_predictions.append(predict(im.convert('RGB'), model, processor, tokenizer))
tuned_seconds = time.time() - t0; tuned = [normalize_text(p) for p in tuned_predictions]
summary = {
    'model_id':MODEL_ID, 'model_revision':MODEL_REVISION, 'gpu':torch.cuda.get_device_name(0),
    'vram_gb':round(VRAM_GB,2), 'load_in_4bit':LOAD_IN_4BIT,
    'counts':{s:len(v) for s,v in records.items()},
    'baseline':{'cer':cer(refs,baseline),'wer':wer(refs,baseline),'seconds':baseline_seconds},
    'finetuned':{'cer':cer(refs,tuned),'wer':wer(refs,tuned),'seconds':tuned_seconds},
    'best_validation_loss':trainer.state.best_metric, 'train_runtime_seconds':train_result.metrics.get('train_runtime'),
}
print(f"Held-out n={len(refs)} | CER {summary['baseline']['cer']:.4f} → {summary['finetuned']['cer']:.4f} | WER {summary['baseline']['wer']:.4f} → {summary['finetuned']['wer']:.4f}")
print('OCR similarity only; this does not establish clinical or medication correctness.')

In [ ]:
# Save private evidence to the ephemeral runtime. No Hub upload and no Google Drive sync.
import json, csv, shutil
OUT = Path('/content/medocr_results')
if OUT.exists(): shutil.rmtree(OUT)
(OUT/'lora_adapter').mkdir(parents=True)
model.save_pretrained(OUT/'lora_adapter'); tokenizer.save_pretrained(OUT/'lora_adapter'); processor.save_pretrained(OUT/'lora_adapter')
(OUT/'report.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
with (OUT/'per_image_results.csv').open('w', newline='', encoding='utf-8-sig') as f:
    fields=['filename','reference','baseline_ocr','finetuned_ocr','baseline_cer','finetuned_cer','baseline_wer','finetuned_wer']
    writer=csv.DictWriter(f, fieldnames=fields); writer.writeheader()
    for row, ref_raw, p0, p1, ref, b, t in zip(test_rows,[r['text'] for r in test_rows],baseline_predictions,tuned_predictions,refs,baseline,tuned):
        writer.writerow({'filename':Path(row['image_path']).name,'reference':ref_raw,'baseline_ocr':p0,'finetuned_ocr':p1,
            'baseline_cer':cer(ref,b),'finetuned_cer':cer(ref,t),'baseline_wer':wer(ref,b),'finetuned_wer':wer(ref,t)})
archive=shutil.make_archive('/content/medocr_evidence','zip',OUT)
print('Evidence ZIP ready. Its CSV contains transcriptions; store privately and clear the Colab runtime afterward.')
from google.colab import files
files.download(archive)